# Adversarial Debate — Google ADK

**Pattern:** Moderator delegates to proposer + critic sub-agents

```
debate_moderator
  ├── travel_proposer  (argues FOR)
  └── travel_critic    (argues AGAINST)
```

Moderator orchestrates the debate, then synthesizes both arguments into a scored verdict.

In [ ]:
import asyncio, os
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
print("✓ Ready")

In [ ]:
proposer_agent = Agent(name="travel_proposer", model="gemini-2.0-flash",
    description="Makes the strongest case FOR a travel claim.",
    instruction="Argue FOR the claim with 3 specific arguments, 150 words. Label: 'FOR: [argument]'",
    tools=[])

critic_agent = Agent(name="travel_critic", model="gemini-2.0-flash",
    description="Argues AGAINST a travel claim and finds weaknesses.",
    instruction="Argue AGAINST the claim. Challenge specific points. 150 words. Label: 'AGAINST: [argument]'",
    tools=[])

moderator_agent = Agent(name="debate_moderator", model="gemini-2.0-flash",
    description="Moderates travel debates and delivers scored verdicts.",
    instruction="""Moderate the debate:\n1. Get FOR argument from travel_proposer.\n2. Get AGAINST argument from travel_critic.\n3. Deliver verdict:\n   ## Debate: [Claim]\n   ### FOR / ### AGAINST / ### Verdict (scores, stronger side, conclusion)""",
    sub_agents=[proposer_agent, critic_agent])

print(f"Moderator + {len(moderator_agent.sub_agents)} debaters")

In [ ]:
async def run_debate(claim: str) -> str:
    ss = InMemorySessionService()
    session = await ss.create_session(app_name="debate_moderator", user_id="u1")
    runner = InMemoryRunner(agent=moderator_agent, session_service=ss)
    final = ""
    async for event in runner.run_async(user_id=session.user_id, session_id=session.id,
        new_message=genai_types.Content(role="user", parts=[genai_types.Part(text=f"Debate: '{claim}'")])):
        if event.is_final_response() and event.content:
            for p in event.content.parts:
                if p.text: final += p.text
    return final

result = await run_debate("Tokyo is the best travel destination for a one-week trip")
print(result)

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| Sub-agents with opposing instructions | ADK sub-agent instructions define their debate persona |
| Moderator synthesizes verdict | Same pattern as Orchestrator — delegate then synthesize |

### Adversarial Debate: All 4 Frameworks
| Framework | How Debate is Structured | Verdict Format |
|---|---|---|
| LangChain | 3 LCEL chains | Free text |
| LangGraph | 3 nodes + optional loop edge | State-tracked |
| CrewAI | 3 agents + `DebateVerdict` schema | Pydantic object |
| ADK | Moderator + 2 sub-agents | LLM synthesized |

### Next: [07-Reflexion →](../../07-Reflexion/LangChain/reflexion.ipynb)